In [1]:
import datasets


/root/miniconda3/envs/verl/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
prompt_template = r"""Solve the following math problem step by step. The last line of your response should be of the form Answer: $Answer (without quotes) where $Answer is the answer to the problem. Do not wrap $Answer with \boxed{}.

current question: {{question}}

Below are two examples for format reference.
Example question 1: Solve for x: 3x - 5 = 16.

Response:
Add 5 to both sides: 3x = 21.
Divide both sides by 3: x = 7.
Answer: 7

Example question 2: A jacket costs $80 and is on sale for 25% off. What is the sale price?

Response:
25% of 80 is 0.25 × 80 = 20.
Subtract the discount from the original price: 80 − 20 = 60.
Answer: 60

Solve the current question. Remember to put your answer on its own line after "Answer:".
"""

In [8]:
from pathlib import Path
import pandas as pd

# Load both configs
AIME2025_I = datasets.load_dataset("opencompass/AIME2025", name="AIME2025-I", split="test")
AIME2025_II = datasets.load_dataset("opencompass/AIME2025", name="AIME2025-II", split="test")

# Merge and duplicate 32x
merged = datasets.concatenate_datasets([AIME2025_I, AIME2025_II])
duped = datasets.concatenate_datasets([merged] * 32)

# Convert to proper DAPO format
def convert_to_dapo_format(ds):
    data = []
    for item in ds:
        # Create prompt in the expected format
        prompt_content = prompt_template.replace("{{question}}", item['question'])
        
        prompt = [{"content": prompt_content, "role": "user"}]
        
        # Create reward_model dict
        reward_model = {
            "ground_truth": str(item['answer']),
            "style": "rule-lighteval/AIME2025"
        }
        
        # Create extra_info dict
        extra_info = {
            "index": len(data),
            "raw_problem": item['question'],
            "split": None
        }
        
        data.append({
            "data_source": "aime2025",
            "prompt": prompt,
            "ability": "AIME",
            "reward_model": reward_model,
            "extra_info": extra_info
        })
    
    return pd.DataFrame(data)

# Convert to proper format and save
df_dapo_format = convert_to_dapo_format(duped)
out_path = Path("~/code_space/verl/data/fixprompt-aime-2025.parquet").expanduser()
out_path.parent.mkdir(parents=True, exist_ok=True)
df_dapo_format.to_parquet(str(out_path))

len(AIME2025_I), len(AIME2025_II), len(merged), len(duped), len(df_dapo_format), out_path


(15,
 15,
 30,
 960,
 960,
 PosixPath('/root/code_space/verl/data/fixprompt-aime-2025.parquet'))

In [9]:

# Convert MATH-500 to DAPO format
def convert_math500_to_dapo_format(ds):
    data = []
    for item in ds:
        # Create prompt in the expected format
        prompt_content = f"Solve the following math problem step by step. The last line of your response should be of the form Answer: $Answer (without quotes) where $Answer is the answer to the problem.\n\n{item['problem']}\n\nRemember to put your answer on its own line after \"Answer:\"."
        
        prompt = [{"content": prompt_content, "role": "user"}]
        
        # Create reward_model dict
        reward_model = {
            "ground_truth": str(item['answer']),
            "style": "rule-lighteval/MATH500"
        }
        
        # Create extra_info dict
        extra_info = {
            "index": len(data),
            "raw_problem": item['problem'],
            "split": None,
            "subject": item.get('subject', 'Unknown'),
            "level": item.get('level', 'Unknown')
        }
        
        data.append({
            "data_source": "math500",
            "prompt": prompt,
            "ability": "MATH",
            "reward_model": reward_model,
            "extra_info": extra_info
        })
    
    return pd.DataFrame(data)

ds = datasets.load_dataset("HuggingFaceH4/MATH-500", split="test")
# Convert MATH-500 dataset to proper format and save
df_math500_dapo = convert_math500_to_dapo_format(ds)
out_path_math500 = Path("~/code_space/verl/data/fixprompt-math-500.parquet").expanduser()
out_path_math500.parent.mkdir(parents=True, exist_ok=True)
df_math500_dapo.to_parquet(str(out_path_math500))

print(f"MATH-500 converted: {len(df_math500_dapo)} samples")
print(f"Saved to: {out_path_math500}")
df_math500_dapo.head(1)


MATH-500 converted: 500 samples
Saved to: /root/code_space/verl/data/fixprompt-math-500.parquet


,data_source,prompt,ability,reward_model,extra_info
0,math500,[{'content': 'Solve the following math problem...,MATH,"{'ground_truth': '\left( 3, \frac{\pi}{2} \rig...","{'index': 0, 'raw_problem': 'Convert the point..."


In [5]:
import pandas as pd
from pathlib import Path

# Load the three parquet files
aime_2025 = pd.read_parquet("/home/hieunt/verl/data/aime-2025.parquet")
aime_2024 = pd.read_parquet("/home/hieunt/verl/data/aime-2024.parquet")
dapo_math = pd.read_parquet("/home/hieunt/verl/data/dapo-math-17k.parquet")

def print_row_details(df, name, num_rows=2):
    print(f"\n=== {name} ===")
    print(f"Shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    
    for i in range(min(num_rows, len(df))):
        print(f"\n--- Row {i} ---")
        row = df.iloc[i]
        for col in df.columns:
            value = row[col]
            if isinstance(value, str) and len(value) > 200:
                print(f"{col}: {value[:200]}... (truncated, full length: {len(value)})")
            else:
                print(f"{col}: {value}")
    print("\n" + "="*80)

print_row_details(aime_2025, "AIME 2025")
print_row_details(aime_2024, "AIME 2024") 
print_row_details(dapo_math, "DAPO MATH 17K")


FileNotFoundError: [Errno 2] No such file or directory: '/home/hieunt/verl/data/aime-2025.parquet'